# 02 · Efficient student and proposed structural secant distillation
The teacher is EfficientNet-B4; the default deployed student is MobileNetV3-Small.
FreqBlender already has no inference-time parser, so acceleration comes from the compact architecture.

Proposed extension: transfer the teacher's local logit response to FPNet-defined
structural-frequency changes, and suppress noise response when the teacher supports doing so.
For $D_{h,k}=.01[a_h(x_k)-a_h(x)]/\max(\mathrm{RMS}(x_k-x),.001)$:
$$L_{SSD}=E[w\,\mathrm{Huber}(D_{S,str}-D_{T,str})+
\gamma w e^{-|D_{T,noi}|}\mathrm{Huber}(D_{S,noi})].$$
Train with $CE+\alpha\tau^2 KL(T\Vert S)+\beta L_{SSD}$.
See `docs/METHOD.md` for the intervention definition, gates, derivation and prior art.
This is a testable research proposal; novelty and improvement are not established.


This PNG dataset has no supplied video/source grouping. All video-named metrics, validation scores, thresholds and bootstrap groups below operate on individual images. They must be reported as image-level results, not source-video results. Existing folder splits are preserved.


## Runtime and embedded project setup
For training, select **Runtime → Change runtime type → GPU**. This cell unpacks
the complete project source into `/kaggle/working/freqblender_project` and installs support
dependencies. It keeps Kaggle's matched PyTorch/torchvision CUDA installation.
No GitHub fork or separate project ZIP upload is needed.


## Kaggle input and writable outputs
Attach `delno002/split-ffpp` to every notebook. Its root must contain
train/real, train/fake, val/real, val/fake, test/real and test/fake PNG folders.
Existing splits are preserved. Images are read directly from the attached dataset.
Save notebook outputs before ending a session. For later stages, attach the previous
saved output and set PREVIOUS_RUNS to its `freqblender/runs` directory.


In [ ]:
import shutil
import sys, subprocess
from pathlib import Path
import torch


FFPP_ROOT = Path("/kaggle/input/split-ffpp")  # EDIT if Kaggle mounts a different path
DRIVE_PROJECT = Path("/kaggle/working/freqblender")  # legacy variable name, local Kaggle path
DATA_ROOT = DRIVE_PROJECT / "prepared"
RUN_ROOT = DRIVE_PROJECT / "runs"
MANIFEST = DATA_ROOT / "manifest.csv"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PREVIOUS_RUNS = None  # e.g. Path("/kaggle/input/previous-output/freqblender/runs")
if PREVIOUS_RUNS is not None:
    # Copy missing files only, preserving newer checkpoints from this session.
    previous = Path(PREVIOUS_RUNS)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    for source in previous.rglob("*"):
        target = RUN_ROOT / source.relative_to(previous)
        if source.is_file() and not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
from freqblender.data import build_image_manifest
MANIFEST = build_image_manifest(FFPP_ROOT, MANIFEST)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(DRIVE_PROJECT / "environment.txt").write_text(versions)
print("Manifest:", MANIFEST)
print("Using image-level samples; original video/source identities are unavailable.")


In [ ]:
assert torch.cuda.is_available(), "Choose a GPU runtime before training."
from freqblender.data import read_manifest
table = read_manifest(MANIFEST)
display(table.groupby(["split", "label"]).size().unstack(fill_value=0))
SIZE = 224          # 400 is supported; use SAME size in all stages/ablations
BATCH_SIZE = 4      # real-image batch; proposed stage processes 6x this per forward
WORKERS = 2
SEED = 42
MAX_STEPS = 0       # 0 = full epochs; positive values are execution tests only
AMP = False         # FP32 recommended for small secant response differences
def train_stage(stage, name, epochs, architecture="efficientnet_b4", extra=()):
    command = [sys.executable, "-m", "freqblender.train", "--stage", stage,
               "--manifest", str(MANIFEST), "--output", str(RUN_ROOT/name),
               "--architecture", architecture, "--size", str(SIZE),
               "--batch-size", str(BATCH_SIZE), "--workers", str(WORKERS),
               "--epochs", str(epochs), "--seed", str(SEED),
               "--max-steps", str(MAX_STEPS), "--resume"]
    if AMP:
        command += ["--amp"]
    subprocess.run(command + list(map(str, extra)), check=True)
    return RUN_ROOT/name/"best.pt"


In [ ]:
STUDENT_ARCH = "mobilenet_v3_small"  # alternative: efficientnet_b0
STUDENT_EPOCHS = 30
PARSER = RUN_ROOT / "parser" / "best.pt"
TEACHER = RUN_ROOT / "teacher_freqblender" / "best.pt"
assert PARSER.exists() and TEACHER.exists(), "Complete notebook 01 first."
common_args = ["--parser", PARSER, "--teacher", TEACHER, "--temperature", 4,
               "--kd-weight", .5, "--strength", .25, "--epsilon", .02]


## Controlled ablations
All students share initialization seed, crop resolution, training schedule, and
FreqBlender augmentation. CE and ordinary KD isolate architectural and standard
distillation effects. Structural-only isolates the additional noise term.
Training all four takes longer; run each in separate Kaggle sessions if needed.
Do not choose a variant based on test scores.


In [ ]:
train_stage("student", "student_ce", STUDENT_EPOCHS, STUDENT_ARCH,
            extra=common_args + ["--mode", "ce"])


In [ ]:
train_stage("student", "student_kd", STUDENT_EPOCHS, STUDENT_ARCH,
            extra=common_args + ["--mode", "kd"])


In [ ]:
train_stage("student", "student_structure", STUDENT_EPOCHS, STUDENT_ARCH,
            extra=common_args + ["--mode", "proposed", "--noise-weight", 0])


In [ ]:
train_stage("student", "student_proposed", STUDENT_EPOCHS, STUDENT_ARCH,
            extra=common_args + ["--mode", "proposed", "--noise-weight", .25])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
rows = []
for name in ("student_ce", "student_kd", "student_structure", "student_proposed"):
    history = pd.read_json(RUN_ROOT/name/"history.json")
    plt.plot(history.epoch, history.validation_score, label=name)
    rows.append({"run": name, "best_validation_image_auc": history.validation_score.max()})
plt.xlabel("Epoch"); plt.ylabel("Validation image AUC"); plt.legend(); plt.grid(alpha=.2)
display(pd.DataFrame(rows))


Repeat with seeds 43 and 44 in separate run roots and matched teachers to assess variability.
Tune the secant weight/strength only on validation. Report unsuccessful variants too.
The proposed student has exactly the same inference architecture as the CE/KD students.
Open **03_evaluate_and_export.ipynb** once training and model choices are fixed.
